In [1]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("S3 Access Example") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .getOrCreate()
from pyspark.sql.functions import col, from_unixtime, avg, count
from dotenv import load_dotenv

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/20 19:27:00 WARN Utils: Your hostname, iamaral-zorin, resolves to a loopback address: 127.0.1.1; using 192.168.15.12 instead (on interface wlo1)
26/06/20 19:27:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/iamaral/Documents/dev-projects/pipeline-kafka/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/iamaral/.ivy2.5.2/cache
The jars for the packages stored in: /home/iamaral/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6b579032-04ea-4ea5-8cce-94c50f2eb742;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wil

In [2]:
# -----------------------
# 1. Carregar Variáveis de Ambiente do .env_kafka_connect
# -----------------------
load_dotenv('../.env_kafka_connect')  # Carregar o arquivo .env

aws_access_key = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_key = os.getenv("AWS_SECRET_ACCESS_KEY")

aws_region = "sa-east-1"

In [3]:
# -----------------------
# 2. Inicializar a Spark Session com Configuração S3
# -----------------------

# Caminho local dos arquivos JAR no mesmo diretório do notebook
# Obter caminho absoluto dos JARs
current_dir = os.path.join(os.path.dirname(os.getcwd()), "jar")
hadoop_aws_jar = os.path.join(current_dir, "hadoop-aws-3.3.4.jar")
aws_sdk_jar = os.path.join(current_dir, "aws-java-sdk-bundle-1.12.262.jar")

jars_path = f"{hadoop_aws_jar},{aws_sdk_jar}"

spark = SparkSession.builder \
    .appName("ETL Pipeline - S3 Integration") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars", jars_path) \
    .getOrCreate()


spark._jsc.hadoopConfiguration().set("fs.s3a.threads.keepalivetime", "60")
spark._jsc.hadoopConfiguration().set("fs.s3a.connection.timeout", "200000")
spark._jsc.hadoopConfiguration().set("fs.s3a.connection.establish.timeout", "50000")
spark._jsc.hadoopConfiguration().set("fs.s3a.multipart.purge.age", "86400")
spark._jsc.hadoopConfiguration().set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

# Configuração do acesso ao S3
spark._jsc.hadoopConfiguration().set("fs.s3a.access.key", aws_access_key)
spark._jsc.hadoopConfiguration().set("fs.s3a.secret.key", aws_secret_key)
spark._jsc.hadoopConfiguration().set("fs.s3a.endpoint", f"s3.{aws_region}.amazonaws.com")
spark._jsc.hadoopConfiguration().set("fs.s3a.connection.ssl.enabled", "true")
spark._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")

26/06/20 19:27:05 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
# -----------------------
# 3. Ler os Dados Brutos - Camada Bronze
# não se esqueça de alterar o nome do seu bucket
# -----------------------

bronze_path = "s3a://amzn-s3-ia-01/raw-data/ipca/kafka/"

# Ler os dados do S3 e testar a conexão
try:
    df_bronze = spark.read.json(bronze_path)
    print("Leitura bem-sucedida!")
    df_bronze.show()
except Exception as e:
    print(f"Erro ao acessar o S3: {e}")


Leitura bem-sucedida!
+-----------+-------------+---------------+-----------+-------------+------------+----+----------+-------------+---------+
|CompraManha|    Data_Base|Data_Vencimento|PUBaseManha|PUCompraManha|PUVendaManha|Tipo|VendaManha|    dt_update|partition|
+-----------+-------------+---------------+-----------+-------------+------------+----+----------+-------------+---------+
|      10.26|1780444800000|  1786752000000|    4623.77|      4628.09|     4623.77|IPCA|     10.38|1781891097340|        0|
|      10.06|1780358400000|  1786752000000|    4622.98|      4626.54|     4622.98|IPCA|     10.18|1781891097340|        0|
|       7.65|1780272000000|  2062800000000|    2424.26|      2449.52|     2424.26|IPCA|      7.77|1781891097333|        0|
|      10.04|1780272000000|  1786752000000|    4620.64|      4624.22|     4620.64|IPCA|     10.16|1781891097333|        0|
|      10.23|1779667200000|  1786752000000|    4604.73|      4608.42|     4604.73|IPCA|     10.35|1781891097336|     

In [5]:
# -----------------------
# 4. Tratamento dos Dados - Camada Silver
# -----------------------
# Remover duplicações e converter timestamps para datas legíveis
df_silver = df_bronze.dropDuplicates()

# Tratar timestamps e converter para formato legível
df_silver = df_silver.withColumn("Data_Vencimento", from_unixtime(col("Data_Vencimento") / 1000, "yyyy-MM-dd")) \
                     .withColumn("Data_Base", from_unixtime(col("Data_Base") / 1000, "yyyy-MM-dd")) \
                     .withColumn("dt_update", from_unixtime(col("dt_update") / 1000, "yyyy-MM-dd HH:mm:ss"))

# Tratar valores nulos
df_silver = df_silver.fillna({
    "PUCompraManha": 0,
    "PUVendaManha": 0,
    "PUBaseManha": 0
})

# Visualizar os dados transformados
print("Dados Transformados (Silver):")
df_silver.show(truncate=False)

# Salvar os dados limpos no S3 em formato Parquet
silver_path = "s3a://amzn-s3-ia-01/processed-data/ipca/silver/"
df_silver.write.mode("overwrite").parquet(silver_path)

Dados Transformados (Silver):


+-----------+----------+---------------+-----------+-------------+------------+----+----------+-------------------+---------+
|CompraManha|Data_Base |Data_Vencimento|PUBaseManha|PUCompraManha|PUVendaManha|Tipo|VendaManha|dt_update          |partition|
+-----------+----------+---------------+-----------+-------------+------------+----+----------+-------------------+---------+
|10.71      |2026-06-07|2026-08-14     |4627.38    |4630.99      |4627.38     |IPCA|10.83     |2026-06-19 14:44:57|0        |
|6.93       |2025-11-16|2045-05-14     |1219.71    |1246.93      |1219.71     |IPCA|7.05      |2026-06-19 14:44:57|0        |
|7.46       |2026-01-12|2035-05-14     |2328.71    |2353.84      |2328.71     |IPCA|7.58      |2026-06-19 14:44:57|0        |
|7.57       |2026-02-19|2032-08-14     |2856.14    |2878.95      |2856.14     |IPCA|7.69      |2026-06-19 14:44:57|0        |
|10.04      |2026-05-31|2026-08-14     |4620.64    |4624.22      |4620.64     |IPCA|10.16     |2026-06-19 14:44:57|0  

26/06/20 19:39:08 WARN Base64: JAXB is unavailable. Will fallback to SDK implementation which may be less performant.If you are using Java 9+, you will need to include javax.xml.bind:jaxb-api as a dependency.
26/06/20 19:39:10 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers
26/06/20 19:39:10 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 84,44% for 9 writers
26/06/20 19:39:11 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


In [6]:
# -----------------------
# 5. Agregação e Métricas - Camada Gold
# -----------------------
# Calcular métricas agregadas
df_gold = df_silver.groupBy("Tipo").agg(
    avg("PUCompraManha").alias("Media_PUCompraManha"),
    avg("PUVendaManha").alias("Media_PUVendaManha"),
    count("*").alias("Total_Registros")
)

# Visualizar as métricas agregadas
print("Dados Agregados (Gold):")
df_gold.show(truncate=False)

# Salvar os dados agregados no S3 em formato Parquet
gold_path = "s3a://amzn-s3-ia-01/analytics/ipca/gold/"
df_gold.write.mode("overwrite").parquet(gold_path)

Dados Agregados (Gold):


+----+-------------------+------------------+---------------+
|Tipo|Media_PUCompraManha|Media_PUVendaManha|Total_Registros|
+----+-------------------+------------------+---------------+
|IPCA|1845.7655519037182 |1830.761155523871 |82759          |
+----+-------------------+------------------+---------------+



In [7]:
spark.stop()